# Github Connectivity project

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

G = nx.read_gexf('exploration_state.gexf')

print(f"Original network:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Connected: {nx.is_connected(G)}")

# If not connected, keep only largest connected component
if not nx.is_connected(G):
    print("\nNetwork is not connected. Extracting largest component...")
    components = list(nx.connected_components(G))
    print(f"  Number of components: {len(components)}")
    
    # Get sizes of all components
    component_sizes = sorted([len(c) for c in components], reverse=True)
    print(f"  Largest component: {component_sizes[0]} nodes")
    print(f"  Second largest: {component_sizes[1] if len(component_sizes) > 1 else 0} nodes")
    
    # Keep largest component
    largest_cc = max(components, key=len)
    G = G.subgraph(largest_cc).copy()
    
    print(f"\nFiltered network (largest component only):")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")
    print(f"  Connected: {nx.is_connected(G)}")
else:
    print("  Graph is already connected!")

# Nogle små stats

In [ ]:
density = nx.density(G)
print(f"Network density: {density:.4f}")

avg_clustering = nx.average_clustering(G.to_undirected())
print(f"Average clustering coefficient: {avg_clustering:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

degrees = [G.degree(node) for node in G.nodes()]
mean_deg = np.mean(degrees)
median_deg = np.median(degrees)

plt.figure(figsize=(8, 6))
plt.hist(degrees, bins=50, alpha=0.7, edgecolor='black', linewidth=0.5, color='steelblue')
plt.axvline(mean_deg, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_deg:.1f}')
plt.axvline(median_deg, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_deg:.1f}')
plt.xlabel('Degree (number of connections)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Degree Distribution in GitHub Developer Network', fontsize=13)
plt.legend(fontsize=10, loc='upper right')
plt.grid(True, alpha=0.3, axis='y')
plt.xlim(0, 500)  # Zoom to where most of the data is

# Add note about long tail
plt.text(0.98, 0.8, f'Long tail extends to {max(degrees)}', 
         transform=plt.gca().transAxes, 
         fontsize=9, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('degree_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
G_undirected = G
assortativity = nx.degree_assortativity_coefficient(G_undirected)
print(f"Degree Assortativity Coefficient: {assortativity:.4f}")

# Shortest path

In [ ]:
import networkx as nx

def shortest_path_between_users(G, source_user, target_user):
    """
    Find the shortest path (in hops) from source_user to target_user.

    Parameters:
        G (nx.Graph): The GitHub collaboration graph.
        source_user (str): Starting username.
        target_user (str): Target username.

    Returns:
        path (list): List of users from source_user to target_user.
    """
    if source_user not in G:
        raise ValueError(f"User '{source_user}' not found in graph")

    if target_user not in G:
        raise ValueError(f"User '{target_user}' not found in graph")

    try:
        path = nx.shortest_path(G, source=source_user, target=target_user)
        return path
    except nx.NetworkXNoPath:
        raise nx.NetworkXNoPath(
            f"No path found from '{source_user}' to '{target_user}'"
        )
    

shortest_path_between_users(G, 'Pakkutaq', 'torvalds')

# Communities

In [ ]:
def detect_structural_communities(G):
    communities = nx.community.louvain_communities(G, seed=123)
    partition = {}
    for comm_id, community in enumerate(communities):
        for node in community:
            partition[node] = comm_id
    modularity = nx.community.modularity(G, communities)
    return communities, partition, modularity

communities, partition, modularity = detect_structural_communities(G)

print(f"Found {len(communities)} communities")
print(f"Modularity: {modularity:.4f}")

# Get community sizes
sizes = [(i, len(comm)) for i, comm in enumerate(communities)]
sizes.sort(key=lambda x: x[1], reverse=True)

print("\nTop 10 communities:")
for rank, (comm_id, size) in enumerate(sizes[:10], 1):
    pct = (size / len(partition)) * 100
    print(f"  {rank}. Community {comm_id}: {size} nodes ({pct:.1f}%)")


# Languages per community

In [ ]:
def analyze_community_languages(communities, partition, repo_data):
    """
    Find dominant languages for each community.
    """
    community_languages = {}
    
    for comm_id in range(len(communities)):
        # Get all users in this community
        users = [node for node, c in partition.items() if c == comm_id]
        
        # Count language bytes for this community
        lang_bytes = {}
        for user in users:
            # Get repos for this user from G graph
            repos_str = G.nodes[user].get("repos", "")
            if repos_str:
                repos = [r.strip() for r in repos_str.split(",")]
                for repo in repos:
                    if repo in repo_data:
                        languages = repo_data[repo].get("languages", {})
                        for lang, bytes_count in languages.items():
                            if lang not in lang_bytes:
                                lang_bytes[lang] = 0
                            lang_bytes[lang] += bytes_count
        
        community_languages[comm_id] = lang_bytes
    
    return community_languages

import json
with open("repo_data.json", "r") as f:
    repo_data = json.load(f)
# Analyze languages per community
comm_langs = analyze_community_languages(communities, partition, repo_data)

# Show top 3 languages for top 10 communities
print("Top languages per community:\n")
for rank, (comm_id, size) in enumerate(sizes[:10], 1):
    langs = comm_langs[comm_id]
    total = sum(langs.values())
    top_langs = sorted(langs.items(), key=lambda x: x[1], reverse=True)[:3]
    
    print(f"{rank}. Community {comm_id} ({size} nodes):")
    for lang, bytes_count in top_langs:
        pct = (bytes_count / total * 100) if total > 0 else 0
        print(f"   - {lang}: {pct:.1f}%")
    print()

In [ ]:
def normalize_languages(comm_langs):
    """
    Group related languages together.
    - Java + Scala + Kotlin -> Java (JVM languages)
    - C + C++ -> C/C++
    """
    language_groups = {
        # JVM languages
        'Java': 'Java',
        'Scala': 'Java',
        'Kotlin': 'Java',
        
        # C family
        'C': 'C/C++',
        'C++': 'C/C++',
    }
    
    normalized_comm_langs = {}
    
    for comm_id, langs in comm_langs.items():
        normalized_langs = {}
        
        for lang, bytes_count in langs.items():
            # Map to group or keep original
            normalized_lang = language_groups.get(lang, lang)
            
            if normalized_lang not in normalized_langs:
                normalized_langs[normalized_lang] = 0
            normalized_langs[normalized_lang] += bytes_count
        
        normalized_comm_langs[comm_id] = normalized_langs
    
    return normalized_comm_langs

# Normalize languages before calculating purity
comm_langs_normalized = normalize_languages(comm_langs)

def calculate_language_purity(comm_langs, top_k=10):
    """
    Calculate purity for each community.
    Purity = percentage of bytes in the dominant language.
    """
    purity_scores = {}
    
    for comm_id, langs in comm_langs.items():
        if not langs:
            continue
        
        total = sum(langs.values())
        max_bytes = max(langs.values())
        purity = (max_bytes / total * 100) if total > 0 else 0
        dominant_lang = max(langs, key=langs.get)
        
        purity_scores[comm_id] = {
            'purity': purity,
            'dominant_language': dominant_lang,
            'total_bytes': total
        }
    
    return purity_scores

# Use normalized languages
purity_scores = calculate_language_purity(comm_langs_normalized)

# Show purity for top communities
print("\nLanguage Purity for Top  Communities (with language grouping):")
print(f"{'Community':<15} {'Size':<10} {'Dominant Lang':<20} {'Purity'}")
print("-" * 70)

for rank, (comm_id, size) in enumerate(sizes[:5], 1):
    if comm_id in purity_scores:
        info = purity_scores[comm_id]
        print(f"Community {comm_id:<5} {size:<10} {info['dominant_language']:<20} {info['purity']:.1f}%")

# Calculate average purity across top communities
top_10_ids = [comm_id for comm_id, _ in sizes]
avg_purity = np.mean([purity_scores[cid]['purity'] for cid in top_10_ids if cid in purity_scores])
print(f"\nAverage purity across communities: {avg_purity:.1f}%")

In [ ]:
def analyze_community_centrality(G, partition, top_communities, top_n=5):
    """
    Calculate centrality within each community's subgraph.
    Returns top N developers by degree and betweenness for each community.
    """
    community_leaders = {}
    
    for comm_id, size in top_communities:
        print(f"\nAnalyzing Community {comm_id} ({size} nodes)...")
        
        # Get subgraph for this community
        comm_nodes = [node for node, c in partition.items() if c == comm_id]
        subgraph = G.subgraph(comm_nodes).copy()
        
        # Calculate centralities (fast on subgraphs!)
        degree_cent = nx.degree_centrality(subgraph)
        betweenness_cent = nx.betweenness_centrality(subgraph)
        
        # Get top N
        top_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:top_n]
        top_between = sorted(betweenness_cent.items(), key=lambda x: x[1], reverse=True)[:top_n]
        
        community_leaders[comm_id] = {
            'size': size,
            'top_degree': top_degree,
            'top_betweenness': top_between
        }
        
        print(f"  Top {top_n} by degree: {[user for user, _ in top_degree]}")
        print(f"  Top {top_n} by betweenness: {[user for user, _ in top_between]}")
    
    return community_leaders

# Analyze top 5 communities
top_5_communities = sizes[:5]
community_leaders = analyze_community_centrality(G, partition, top_5_communities, top_n=5)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get top 5 communities
top_5 = sizes[:5]
comm_ids = [cid for cid, _ in top_5]

# Prepare data
labels = [f"Community {cid}" for cid in comm_ids]
all_languages = set()
for cid in comm_ids:
    all_languages.update(comm_langs_normalized[cid].keys())

# Get top languages across all communities for coloring
lang_totals = {}
for cid in comm_ids:
    for lang, bytes in comm_langs_normalized[cid].items():
        lang_totals[lang] = lang_totals.get(lang, 0) + bytes

top_langs = sorted(lang_totals.keys(), key=lambda x: lang_totals[x], reverse=True)[:8]
other_langs = set(all_languages) - set(top_langs)

# Build percentage data
data = {lang: [] for lang in top_langs}
data['Other'] = []

for cid in comm_ids:
    total = sum(comm_langs_normalized[cid].values())
    for lang in top_langs:
        pct = (comm_langs_normalized[cid].get(lang, 0) / total * 100) if total > 0 else 0
        data[lang].append(pct)
    
    # Other category
    other_sum = sum(comm_langs_normalized[cid].get(lang, 0) for lang in other_langs)
    other_pct = (other_sum / total * 100) if total > 0 else 0
    data['Other'].append(other_pct)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Colors for languages
colors = {
    'Python': '#3776ab',
    'Java': '#b07219',
    'C/C++': '#555555',
    'TypeScript': '#2b7489',
    'JavaScript': '#f1e05a',
    'PHP': '#4F5D95',
    'Go': '#00ADD8',
    'Ruby': '#701516',
    'Other': '#cccccc'
}

# Create stacked bars
bottom = np.zeros(len(comm_ids))
for lang in top_langs + ['Other']:
    values = data[lang]
    ax.barh(labels, values, left=bottom, label=lang, 
            color=colors.get(lang, '#999999'), edgecolor='white', linewidth=0.5)
    bottom += values

ax.set_xlabel('Percentage of Language Bytes (%)', fontsize=12)
ax.set_title('Language Distribution in Top 5 Communities', fontsize=14, fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.1, 0.5), fontsize=10)
ax.set_xlim(0, 100)

# Add purity percentages as text
for i, cid in enumerate(comm_ids):
    purity = purity_scores[cid]['purity']
    ax.text(102, i, f"{purity:.1f}%", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('community_language_alignment.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Semantics analysis

In [ ]:
def extract_readme_texts(repo_data):
    """
    Extract all README texts from repo_data.
    
    Returns:
        list of strings (one per repo with a README)
    """
    readme_texts = []
    
    for repo, data in repo_data.items():
        readme = data.get("readme")
        if readme:
            readme_texts.append(readme)
    
    print(f"Found {len(readme_texts)} repos with READMEs")
    return readme_texts

readmes = extract_readme_texts(repo_data)

# Wordclouds

In [ ]:
from wordcloud import STOPWORDS
import re

def clean_readme_text(text):
    """
    Clean a single README text.
    """
    if not text:
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove markdown links [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    
    # Remove code blocks (```)
    text = re.sub(r'```[\s\S]*?```', '', text)
    
    # Remove inline code (`)
    text = re.sub(r'`[^`]*`', '', text)
    
    # Remove special characters but keep spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def clean_readmes(readme_texts):
    """
    Clean all README texts.
    """
    cleaned = [clean_readme_text(text) for text in readme_texts]
    print(f"Cleaned {len(cleaned)} READMEs")
    return cleaned

# Clean the readmes
cleaned_readmes = clean_readmes(readmes)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def create_wordcloud(texts, title="Word Cloud"):
    """
    Create a word cloud from a list of texts.
    
    Parameters:
        texts: list of strings
        title: title for the plot
    """
    # Combine all texts
    combined_text = " ".join(texts)
    
    # Create word cloud
    wordcloud = WordCloud(
        width=800, 
        height=400,
        background_color='white',
        max_words=100
    ).generate(combined_text)
    
    # Display
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Create overall word cloud
create_wordcloud(cleaned_readmes, "All READMEs Word Cloud")

# TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_community_readmes(community_id, partition, repo_data):
    """
    Collect all README texts for users in a specific community.
    """
    # Get users in this community
    users = [node for node, comm in partition.items() if comm == community_id]
    
    # Collect their READMEs
    readmes = []
    for user in users:
        repos_str = G.nodes[user].get("repos", "")
        if repos_str:
            repos = [r.strip() for r in repos_str.split(",")]
            for repo in repos:
                if repo in repo_data:
                    readme = repo_data[repo].get("readme")
                    if readme:
                        readmes.append(readme)
    
    return readmes

def analyze_community_tfidf(community_id, partition, repo_data, top_n=10):
    """
    Find top TF-IDF terms for a specific community.
    """
    # Get READMEs for this community
    readmes = get_community_readmes(community_id, partition, repo_data)
    
    if not readmes:
        return []
    
    # Clean the READMEs
    cleaned = [clean_readme_text(text) for text in readmes]
    
    # Combine into one document for this community
    community_text = " ".join(cleaned)
    
    return community_text, len(readmes)

# Collect texts for top 5 communities
print("Collecting README texts for top communities...\n")

community_texts = {}
for rank, (comm_id, size) in enumerate(sizes[:5], 1):
    text, readme_count = analyze_community_tfidf(comm_id, partition, repo_data)
    community_texts[comm_id] = text
    print(f"Community {comm_id}: {readme_count} READMEs")

In [ ]:
def compare_communities_tfidf(community_texts, top_n=15):
    """
    Use TF-IDF to find distinctive words for each community.
    """
    # Prepare documents (one per community)
    comm_ids = list(community_texts.keys())
    documents = [community_texts[cid] for cid in comm_ids]
    
    # Custom stopwords
    custom_stopwords = list(STOPWORDS)
    custom_stopwords.extend([
        'use', 'used', 'using', 'run', 'file', 'will', 'one', 'two',
        'project', 'new', 'set', 'may', 'also', 'see', 'get', 'make',
        'example', 'install', 'start', 'first', 'need', 'add', 'name',
        'following', 'via', 'note', 'number', 'type', 'value', 'result',
        'allow', 'include', 'provide', 'contain', 'based', 'support'
    ])
    
    # Compute TF-IDF
    vectorizer = TfidfVectorizer(
        max_features=1000,
        stop_words=custom_stopwords,
        min_df=1,
        ngram_range=(1, 2)  # Include bigrams
    )
    
    tfidf_matrix = vectorizer.fit_transform(documents)
    feature_names = vectorizer.get_feature_names_out()
    
    # Get top terms for each community
    print(f"{'Community':<15} {'Top Distinctive Terms'}")
    print("=" * 80)
    
    for idx, comm_id in enumerate(comm_ids):
        # Get TF-IDF scores for this community
        scores = tfidf_matrix[idx].toarray()[0]
        top_indices = scores.argsort()[-top_n:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        
        # Get dominant language for context
        if comm_id in comm_langs_normalized:
            langs = comm_langs_normalized[comm_id]
            top_lang = max(langs, key=langs.get)
        else:
            top_lang = "Unknown"
        
        print(f"\nCommunity {comm_id:<5} ({top_lang}):")
        print(f"  {', '.join(top_terms)}")

# Run TF-IDF comparison
compare_communities_tfidf(community_texts, top_n=15)

In [ ]:
def create_community_wordclouds(community_texts, struct_comm_langs):
    """
    Create word clouds for each community.
    """
    # Custom stopwords
    custom_stopwords = STOPWORDS.copy()
    custom_stopwords.update([
        'use', 'used', 'using', 'run', 'file', 'will', 'one', 'two',
        'project', 'new', 'set', 'may', 'also', 'see', 'get', 'make',
        'example', 'install', 'start', 'first', 'need', 'add', 'name',
        'following', 'via', 'note', 'number', 'type', 'value', 'result',
        'allow', 'include', 'provide', 'contain', 'based', 'support'
    ])
    
    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (comm_id, text) in enumerate(community_texts.items()):
        # Get dominant language
        if comm_id in comm_langs_normalized:
            langs = comm_langs_normalized[comm_id]
            top_lang = max(langs, key=langs.get)
            purity = (langs[top_lang] / sum(langs.values()) * 100)
        else:
            top_lang = "Unknown"
            purity = 0
        
        # Create wordcloud
        wordcloud = WordCloud(
            width=600,
            height=400,
            background_color='white',
            max_words=50,
            stopwords=custom_stopwords,
            colormap='viridis'
        ).generate(text)
        
        # Plot
        axes[idx].imshow(wordcloud, interpolation='bilinear')
        axes[idx].axis('off')
        axes[idx].set_title(f'Community {comm_id}\n({top_lang}, {purity:.1f}% purity)', 
                           fontsize=12, fontweight='bold')
    
    # Hide extra subplot if we have fewer than 6 communities
    if len(community_texts) < 6:
        for idx in range(len(community_texts), 6):
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Create wordclouds
create_community_wordclouds(community_texts, comm_langs_normalized)

# 6 degrees of separation experiments

In [ ]:
import random

def test_small_world(G, num_samples=10000):
    """
    Test small-world phenomenon by finding shortest paths between random node pairs.
    
    Parameters:
        G: NetworkX graph
        num_samples: Number of random pairs to test
    
    Returns:
        list of path lengths
    """
    nodes = list(G.nodes())
    path_lengths = []
    no_path_count = 0
    
    print(f"Testing {num_samples} random node pairs...\n")
    
    for i in range(num_samples):
        # Pick two random nodes
        node1, node2 = random.sample(nodes, 2)
        
        try:
            # Use your existing function
            path = shortest_path_between_users(G, node1, node2)
            path_length = len(path) - 1  # Number of edges
            path_lengths.append(path_length)
            
            if i < 10:  # Show first 10 examples
                print(f"{i+1}. {node1} → {node2}: {path_length} steps")
        
        except (nx.NetworkXNoPath, ValueError):
            no_path_count += 1
            if i < 10:
                print(f"{i+1}. {node1} → {node2}: No path (disconnected)")
    
    # Statistics
    if path_lengths:
        print(f"\n{'='*60}")
        print(f"Results from {len(path_lengths)} connected pairs:")
        print(f"  Average path length: {np.mean(path_lengths):.2f}")
        print(f"  Median path length: {np.median(path_lengths):.1f}")
        print(f"  Min path length: {min(path_lengths)}")
        print(f"  Max path length: {max(path_lengths)}")
    
    return path_lengths

# Run experiment
path_lengths = test_small_world(G, num_samples=len(G.nodes()))